# Part 1 Verification

Execute each Req's SQL file separately, then validate results.
These are the same files that make up Part1_Group11.sql — just run individually for step-by-step verification.

## Prerequisites (Phase 0 must be complete)

**1. WideWorldImporters restored and accessible (source data):**

![WideWorldImporters source ready](prereq-wwi-source-ready.png)

**2. WWI_DM freshly created (empty — no tables):**

![WWI_DM fresh database](prereq-wwi-dm-fresh.png)

If not, run in SSMS: `DROP DATABASE IF EXISTS WWI_DM; CREATE DATABASE WWI_DM;`
See [SETUP-GUIDE](../../part0-setup/SETUP-GUIDE.md) for full Phase 0 instructions.

In [1]:
import pyodbc
import pandas as pd
import re

# Connect to WWI_DM (our Data Mart)
conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=localhost;"
    r"DATABASE=WWI_DM;"
    r"Trusted_Connection=yes;"
)

def sql(query):
    """Run a SELECT query and return results as a DataFrame."""
    return pd.read_sql(query, conn)

def execute(filepath):
    """
    Read a .sql file and execute it batch by batch.
    
    SQL Server uses GO as a batch separator — each GO-separated block 
    is sent to the server as one unit. pyodbc can't handle GO directly, 
    so we split the file on GO and execute each batch separately.
    
    Commits after EACH batch — this is important because:
    - CREATE PROCEDURE must be committed before EXEC can call it
    - If we commit only at the end, the SP doesn't exist yet when WHILE loop runs
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Split on GO (standalone line, case-insensitive, handles \r\n)
    batches = re.split(r'(?m)^\s*GO\s*$', content)
    cursor = conn.cursor()
    executed = 0
    
    for i, batch in enumerate(batches):
        # Skip if batch is only comments/whitespace (no actual SQL)
        lines = [l for l in batch.strip().splitlines() 
                 if l.strip() and not l.strip().startswith('--')]
        if not lines:
            continue
        try:
            cursor.execute(batch.strip())
            conn.commit()  # Commit each batch immediately
            executed += 1
            print(f"  Batch {i+1}: OK")
        except Exception as e:
            print(f"  Batch {i+1}: ERROR - {e}")
    
    print(f"Done: {filepath} ({executed} batches executed)")

print("Connected to WWI_DM")

Connected to WWI_DM


## Req 1: Create Tables

Execute create-tables.sql → creates 6 Dim tables + FactSales + FKs + Indexes

In [2]:
execute("../../part1/req1-schema/create-tables.sql")

  Batch 1: OK
  Batch 2: OK
  Batch 3: OK
  Batch 4: OK
  Batch 5: OK
  Batch 6: OK
  Batch 7: OK
  Batch 8: OK
  Batch 9: OK
Done: ../../part1/req1-schema/create-tables.sql (9 batches executed)


### Validate Req 1: Tables exist? (expected: 7)

**SSMS Object Explorer after execution:**

![7 tables created](result-req1-tables-created.png)

In [3]:
sql("SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES ORDER BY TABLE_NAME")

C:\Users\blitz\AppData\Local\Temp\ipykernel_7164\3441237571.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,TABLE_NAME
0,DimCustomers
1,DimDate
2,DimLocation
3,DimProducts
4,DimSalesPeople
5,DimSuppliers
6,FactSales


### Validate Req 1: FKs on FactSales? (expected: 6)

In [4]:
sql("""
    SELECT CONSTRAINT_NAME 
    FROM INFORMATION_SCHEMA.TABLE_CONSTRAINTS 
    WHERE TABLE_NAME = 'FactSales' AND CONSTRAINT_TYPE = 'FOREIGN KEY'
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_7164\3441237571.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,CONSTRAINT_NAME
0,FK_FactSales_DimCustomers
1,FK_FactSales_DimLocation
2,FK_FactSales_DimProducts
3,FK_FactSales_DimSalesPeople
4,FK_FactSales_DimSuppliers
5,FK_FactSales_DimDate


### Validate Req 1: Indexes on FactSales? (expected: 6 non-clustered)

In [5]:
sql("""
    SELECT name AS IndexName, type_desc
    FROM sys.indexes
    WHERE object_id = OBJECT_ID('dbo.FactSales') AND name IS NOT NULL
""")

C:\Users\blitz\AppData\Local\Temp\ipykernel_7164\3441237571.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,IndexName,type_desc
0,IX_FactSales_CustomerKey,NONCLUSTERED
1,IX_FactSales_LocationKey,NONCLUSTERED
2,IX_FactSales_ProductKey,NONCLUSTERED
3,IX_FactSales_SalespersonKey,NONCLUSTERED
4,IX_FactSales_SupplierKey,NONCLUSTERED
5,IX_FactSales_DateKey,NONCLUSTERED


### Validate Req 1: All table structures — columns, types, sizes, nullable

In [6]:
# Full metadata for ALL tables in WWI_DM
for table in ['DimLocation', 'DimCustomers', 'DimProducts', 'DimSalesPeople', 'DimDate', 'DimSuppliers', 'FactSales']:
    print(f"\n=== {table} ===")
    display(sql(f"""
        SELECT COLUMN_NAME, DATA_TYPE, 
               CHARACTER_MAXIMUM_LENGTH AS MaxLen,
               NUMERIC_PRECISION AS NumPrec, 
               NUMERIC_SCALE AS NumScale,
               IS_NULLABLE
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_NAME = '{table}'
        ORDER BY ORDINAL_POSITION
    """))

C:\Users\blitz\AppData\Local\Temp\ipykernel_7164\3441237571.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)



=== DimLocation ===


,COLUMN_NAME,DATA_TYPE,MaxLen,NumPrec,NumScale,IS_NULLABLE
0,LocationKey,int,NaN,10.0,0.0,NO
1,CityName,nvarchar,50.0,NaN,NaN,YES
2,StateProvCode,nvarchar,5.0,NaN,NaN,YES
3,StateProvName,nvarchar,50.0,NaN,NaN,YES
4,CountryName,nvarchar,60.0,NaN,NaN,YES
5,CountryFormalName,nvarchar,60.0,NaN,NaN,YES



=== DimCustomers ===


C:\Users\blitz\AppData\Local\Temp\ipykernel_7164\3441237571.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


...

C:\Users\blitz\AppData\Local\Temp\ipykernel_7164\3441237571.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,COLUMN_NAME,DATA_TYPE,MaxLen,NumPrec,NumScale,IS_NULLABLE
0,SupplierKey,int,NaN,10.0,0.0,NO
1,FullName,nvarchar,100.0,NaN,NaN,YES
2,PhoneNumber,nvarchar,20.0,NaN,NaN,YES
3,FaxNumber,nvarchar,20.0,NaN,NaN,YES
4,WebsiteURL,nvarchar,256.0,NaN,NaN,YES
5,SupplierCategoryName,nvarchar,50.0,NaN,NaN,YES
6,StartDate,date,NaN,NaN,NaN,NO
7,EndDate,date,NaN,NaN,NaN,YES



=== FactSales ===


C:\Users\blitz\AppData\Local\Temp\ipykernel_7164\3441237571.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,COLUMN_NAME,DATA_TYPE,MaxLen,NumPrec,NumScale,IS_NULLABLE
0,CustomerKey,int,None,10,0,NO
1,LocationKey,int,None,10,0,NO
2,ProductKey,int,None,10,0,NO
3,SalespersonKey,int,None,10,0,NO
4,SupplierKey,int,None,10,0,NO
5,DateKey,int,None,10,0,NO
6,Quantity,int,None,10,0,NO
7,UnitPrice,decimal,None,18,2,NO
8,TaxRate,decimal,None,18,3,NO
9,TotalBeforeTax,decimal,None,18,2,NO


### Validate Req 1: All Dims vs source — types, sizes, nullable compatible?

Dim columns must be compatible with source: same type, size >= source, nullable matches.

In [7]:
# Compare source (WideWorldImporters) vs target (WWI_DM) — full metadata
conn_source = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=localhost;"
    r"DATABASE=WideWorldImporters;"
    r"Trusted_Connection=yes;"
    r"MARS_Connection=Yes;"
)

comparisons = [
    ("DimSuppliers", [
        ("FullName", "Purchasing", "Suppliers", "SupplierName"),
        ("PhoneNumber", "Purchasing", "Suppliers", "PhoneNumber"),
        ("FaxNumber", "Purchasing", "Suppliers", "FaxNumber"),
        ("WebsiteURL", "Purchasing", "Suppliers", "WebsiteURL"),
        ("SupplierCategoryName", "Purchasing", "SupplierCategories", "SupplierCategoryName"),
    ]),
    ("DimCustomers", [
        ("CustomerName", "Sales", "Customers", "CustomerName"),
        ("CustomerCategoryName", "Sales", "CustomerCategories", "CustomerCategoryName"),
    ]),
    ("DimProducts", [
        ("ProductName", "Warehouse", "StockItems", "StockItemName"),
        ("ProductColour", "Warehouse", "Colors", "ColorName"),
        ("ProductBrand", "Warehouse", "StockItems", "Brand"),
        ("ProductSize", "Warehouse", "StockItems", "Size"),
    ]),
    ("DimSalesPeople", [
        ("FullName", "Application", "People", "FullName"),
        ("PreferredName", "Application", "People", "PreferredName"),
        ("LogonName", "Application", "People", "LogonName"),
        ("PhoneNumber", "Application", "People", "PhoneNumber"),
        ("FaxNumber", "Application", "People", "FaxNumber"),
        ("EmailAddress", "Application", "People", "EmailAddress"),
    ]),
    ("DimLocation", [
        ("CityName", "Application", "Cities", "CityName"),
        ("StateProvCode", "Application", "StateProvinces", "StateProvinceCode"),
        ("StateProvName", "Application", "StateProvinces", "StateProvinceName"),
        ("CountryName", "Application", "Countries", "CountryName"),
        ("CountryFormalName", "Application", "Countries", "FormalName"),
    ]),
]

def get_col_meta(connection, schema, table, column):
    df = pd.read_sql(f"""
        SELECT DATA_TYPE, CHARACTER_MAXIMUM_LENGTH, 
               NUMERIC_PRECISION, NUMERIC_SCALE, IS_NULLABLE
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = '{schema}' AND TABLE_NAME = '{table}' AND COLUMN_NAME = '{column}'
    """, connection)
    if len(df) == 0:
        return {'DATA_TYPE': '???', 'CHARACTER_MAXIMUM_LENGTH': None, 
                'NUMERIC_PRECISION': None, 'NUMERIC_SCALE': None, 'IS_NULLABLE': '???'}
    return df.iloc[0].to_dict()

def get_dim_meta(connection, table, column):
    df = pd.read_sql(f"""
        SELECT DATA_TYPE, CHARACTER_MAXIMUM_LENGTH,
               NUMERIC_PRECISION, NUMERIC_SCALE, IS_NULLABLE
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_NAME = '{table}' AND COLUMN_NAME = '{column}'
    """, connection)
    if len(df) == 0:
        return {'DATA_TYPE': '???', 'CHARACTER_MAXIMUM_LENGTH': None,
                'NUMERIC_PRECISION': None, 'NUMERIC_SCALE': None, 'IS_NULLABLE': '???'}
    return df.iloc[0].to_dict()

def fmt_type(m):
    t = m['DATA_TYPE']
    if m['CHARACTER_MAXIMUM_LENGTH'] is not None:
        return f"{t}({int(m['CHARACTER_MAXIMUM_LENGTH'])})"
    elif m['NUMERIC_PRECISION'] is not None:
        return f"{t}({int(m['NUMERIC_PRECISION'])},{int(m['NUMERIC_SCALE'])})"
    return t

results = []
for dim_table, mappings in comparisons:
    for dim_col, src_schema, src_table, src_col in mappings:
        s = get_col_meta(conn_source, src_schema, src_table, src_col)
        d = get_dim_meta(conn, dim_table, dim_col)
        
        checks = []
        # Type match
        if s['DATA_TYPE'] != d['DATA_TYPE']:
            checks.append(f"❌ type mismatch")
        # Size check (for string types)
        s_len = s['CHARACTER_MAXIMUM_LENGTH']
        d_len = d['CHARACTER_MAXIMUM_LENGTH']
        if s_len is not None and d_len is not None:
            if d_len < s_len:
                checks.append(f"❌ truncation risk (src={int(s_len)}, dim={int(d_len)})")
        # Precision check (for numeric types)
        if s['NUMERIC_PRECISION'] is not None and d['NUMERIC_PRECISION'] is not None:
            if d['NUMERIC_PRECISION'] < s['NUMERIC_PRECISION'] or d['NUMERIC_SCALE'] < s['NUMERIC_SCALE']:
                checks.append(f"❌ precision loss")
        # Nullable check
        if s['IS_NULLABLE'] == 'NO' and d['IS_NULLABLE'] == 'YES':
            checks.append("⚠️ src=NOT NULL, dim=NULL")
        elif s['IS_NULLABLE'] == 'YES' and d['IS_NULLABLE'] == 'NO':
            checks.append("⚠️ src=NULL, dim=NOT NULL")
        
        status = ' | '.join(checks) if checks else '✅'
        
        results.append({
            'Dim': dim_table,
            'DimColumn': dim_col,
            'Source': f"{src_schema}.{src_table}.{src_col}",
            'SrcType': fmt_type(s),
            'SrcNull': s['IS_NULLABLE'],
            'DimType': fmt_type(d),
            'DimNull': d['IS_NULLABLE'],
            'Check': status
        })

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.DataFrame(results)

C:\Users\blitz\AppData\Local\Temp\ipykernel_7164\2200082829.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"""
C:\Users\blitz\AppData\Local\Temp\ipykernel_7164\2200082829.py:58: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"""


,Dim,DimColumn,Source,SrcType,SrcNull,DimType,DimNull,Check
0,DimSuppliers,FullName,Purchasing.Suppliers.SupplierName,nvarchar(100),NO,nvarchar(100),YES,"⚠️ src=NOT NULL, dim=NULL"
1,DimSuppliers,PhoneNumber,Purchasing.Suppliers.PhoneNumber,nvarchar(20),NO,nvarchar(20),YES,"⚠️ src=NOT NULL, dim=NULL"
2,DimSuppliers,FaxNumber,Purchasing.Suppliers.FaxNumber,nvarchar(20),NO,nvarchar(20),YES,"⚠️ src=NOT NULL, dim=NULL"
3,DimSuppliers,WebsiteURL,Purchasing.Suppliers.WebsiteURL,nvarchar(256),NO,nvarchar(256),YES,"⚠️ src=NOT NULL, dim=NULL"
4,DimSuppliers,SupplierCategoryName,Purchasing.SupplierCategories.SupplierCategoryName,nvarchar(50),NO,nvarchar(50),YES,"⚠️ src=NOT NULL, dim=NULL"
5,DimCustomers,CustomerName,Sales.Customers.CustomerName,nvarchar(100),NO,nvarchar(100),YES,"⚠️ src=NOT NULL, dim=NULL"
6,DimCustomers,CustomerCategoryName,Sales.CustomerCategories.CustomerCategoryName,nvarchar(50),NO,nvarchar(50),YES,"⚠️ src=NOT NULL, dim=NULL"
7,DimProducts,ProductName,Warehouse.StockItems.StockItemName,nvarchar(100),NO,nvarchar(100),YES,"⚠️ src=NOT NULL, dim=NULL"
8,DimProducts,ProductColour,Warehouse.Colors.ColorName,nvarchar(20),NO,nvarchar(20),YES,"⚠️ src=NOT NULL, dim=NULL"
9,DimProducts,ProductBrand,Warehouse.StockItems.Brand,nvarchar(50),YES,nvarchar(50),YES,✅


## Req 2: DimDate Load

Execute dimdate-load.sql → creates DimDate_Load SP + loads 5 years (2012-2016)

In [8]:
# Req 2: Create SP first, then call it 1,827 times from Python
# Can't rely on T-SQL WHILE loop through pyodbc — it truncates

# Step 1: Create the SP
cursor = conn.cursor()
cursor.execute("""
CREATE OR ALTER PROCEDURE dbo.DimDate_Load
    @DateValue DATE
AS
BEGIN
    INSERT INTO dbo.DimDate
    SELECT
        CAST(YEAR(@DateValue) * 10000 + MONTH(@DateValue) * 100 + DAY(@DateValue) AS INT),
        @DateValue,
        YEAR(@DateValue),
        MONTH(@DateValue),
        DAY(@DateValue),
        DATEPART(qq, @DateValue),
        DATEADD(DAY, 1, EOMONTH(@DateValue, -1)),
        EOMONTH(@DateValue),
        DATENAME(mm, @DateValue),
        DATENAME(dw, @DateValue);
END;
""")
conn.commit()
print("SP created: DimDate_Load")

# Step 2: Call SP for each day (Python loop instead of T-SQL WHILE)
from datetime import date, timedelta

start = date(2012, 1, 1)
end = date(2016, 12, 31)
current = start
count = 0

while current <= end:
    cursor.execute("EXEC dbo.DimDate_Load @DateValue = ?", current.isoformat())
    current += timedelta(days=1)
    count += 1

conn.commit()
print(f"Done: {count} dates loaded (expected: 1,827)")

SP created: DimDate_Load
Done: 1827 dates loaded (expected: 1,827)


### Validate Req 2: DimDate row count? (expected: 1,827)

In [9]:
sql("SELECT COUNT(*) AS DimDateRows FROM DimDate")

C:\Users\blitz\AppData\Local\Temp\ipykernel_7164\3441237571.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,DimDateRows
0,1827


### Validate Req 2: Spot check — 2013-01-01 = Tuesday?

In [10]:
sql("SELECT DateKey, DateValue, CYear, CMonth, DayNo, CQtr, MonthName, DayOfWeekName FROM DimDate WHERE DateKey = 20130101")

C:\Users\blitz\AppData\Local\Temp\ipykernel_7164\3441237571.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


,DateKey,DateValue,CYear,CMonth,DayNo,CQtr,MonthName,DayOfWeekName
0,20130101,2013-01-01,2013,1,1,1,January,Tuesday


## Req 3: Compelling Queries

Execute compelling-query.sql → 3 queries. FactSales is empty so 0 rows expected — just verify no SQL errors.

In [11]:
execute("../../part1/req3-query/compelling-query.sql")

  Batch 1: OK
  Batch 2: OK
  Batch 3: OK
  Batch 4: OK
Done: ../../part1/req3-query/compelling-query.sql (4 batches executed)


✅ **Part 1 passed if:**
- Req 1: 7 tables, 6 FKs, 6 indexes, DimSuppliers has correct columns
- Req 2: DimDate = 1,827 rows, 2013-01-01 = Tuesday
- Req 3: Queries executed without SQL errors (0 rows expected until Req 7)